In [4]:
import psycopg2

# Connect to the database
conn = psycopg2.connect("dbname=hn user=julien")

# Open a cursor to perform database operations
cur = conn.cursor()
query = "SELECT id, embedding FROM hn_embeddings WHERE path IS NULL"
                
cur.execute(query)


# Retrieve query results
results = cur.fetchall()
print(len(results))
print(results[:10])

0
[]


In [19]:
conn.reset()
q = """
SELECT
        hn_embeddings.id,
        title,
        url,
        score,
        descendants as comments,
        time,
        path
FROM hn_embeddings
INNER JOIN hn_post ON hn_embeddings.id = hn_post.id
WHERE path IS NOT NULL
ORDER BY embedding <#> (SELECT embedding FROM hn_embeddings WHERE id = {})
LIMIT 1
"""
i = 0
for post in results:
        cur.execute(q.format(post[0]))
        # Set the path to the nearest post path
        path = cur.fetchone()[6]
        cur.execute("UPDATE hn_embeddings SET path = %s WHERE id = %s", (path, post[0]))
        i += 1
        print('\r', i, "/", len(results), end='')

conn.commit()


 3426 / 8426